# StudyVault — CRUD Operations with ChromaDB

### Student Knowledge Management System

This project demonstrates CRUD operations using ChromaDB.

Operations covered:
- Create
- Read
- Update
- Delete
- Semantic Search
- Metadata Filtering
- Upsert

In [6]:
%pip install -q chromadb

In [7]:
import chromadb

print("ChromaDB version:", chromadb.__version__)
print("ChromaDB imported successfully!")

ChromaDB version: 1.5.9
ChromaDB imported successfully!


In [8]:
import chromadb

client = chromadb.PersistentClient(path="./studyvault_db")

print("Client created successfully!")

Client created successfully!


In [9]:
collection = client.get_or_create_collection(
    name="student_knowledge"
)

print("Collection:", collection.name)
print("Document count:", collection.count())


Collection: student_knowledge
Document count: 0


## CREATE — Adding Study Resources

We store learning resources with:
- unique IDs
- document text
- subject metadata
- difficulty metadata

In [10]:
collection.add(
    ids=[
        "study01",
        "study02",
        "study03",
        "study04",
        "study05",
        "study06"
    ],

    documents=[
        "JavaScript functions allow developers to organize reusable blocks of code.",
        "SQL JOIN operations combine records from multiple database tables.",
        "Machine learning algorithms learn patterns from historical data.",
        "React components help developers build reusable user interface elements.",
        "Python lists store multiple values in an ordered and changeable collection.",
        "REST APIs allow applications to communicate using HTTP requests."
    ],

    metadatas=[
        {"subject": "javascript", "level": "beginner"},
        {"subject": "sql", "level": "intermediate"},
        {"subject": "machine_learning", "level": "intermediate"},
        {"subject": "react", "level": "beginner"},
        {"subject": "python", "level": "beginner"},
        {"subject": "backend", "level": "intermediate"}
    ]
)

print("Documents added successfully!")
print("Total documents:", collection.count())

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 56.3MiB/s]


Documents added successfully!
Total documents: 6


## READ — Preview Study Resources

Use `peek()` to quickly inspect a few documents stored in ChromaDB.

In [11]:
preview = collection.peek(limit=3)

print("Preview of StudyVault:\n")

for doc_id, doc, metadata in zip(
    preview["ids"],
    preview["documents"],
    preview["metadatas"]
):
    print(f"ID: {doc_id}")
    print(f"Subject: {metadata['subject']}")
    print(f"Level: {metadata['level']}")
    print(f"Content: {doc}")
    print("-" * 60)

Preview of StudyVault:

ID: study01
Subject: javascript
Level: beginner
Content: JavaScript functions allow developers to organize reusable blocks of code.
------------------------------------------------------------
ID: study02
Subject: sql
Level: intermediate
Content: SQL JOIN operations combine records from multiple database tables.
------------------------------------------------------------
ID: study03
Subject: machine_learning
Level: intermediate
Content: Machine learning algorithms learn patterns from historical data.
------------------------------------------------------------


## READ — Retrieve Resources by ID

Use `get()` when the exact document ID is known.

In [12]:
result = collection.get(
    ids=["study02", "study05"]
)

print("Selected Study Resources:\n")

for doc_id, doc, metadata in zip(
    result["ids"],
    result["documents"],
    result["metadatas"]
):
    print(f"ID: {doc_id}")
    print(f"Subject: {metadata['subject']}")
    print(f"Level: {metadata['level']}")
    print(f"Content: {doc}")
    print("-" * 60)

Selected Study Resources:

ID: study02
Subject: sql
Level: intermediate
Content: SQL JOIN operations combine records from multiple database tables.
------------------------------------------------------------
ID: study05
Subject: python
Level: beginner
Content: Python lists store multiple values in an ordered and changeable collection.
------------------------------------------------------------


## READ — Semantic Search

Search the StudyVault knowledge base using the meaning of a question.

The query does not need to exactly match the words stored in the documents.

In [13]:
results = collection.query(
    query_texts=[
        "How can I combine information from different database tables?"
    ],
    n_results=3
)

print("Semantic Search Results:\n")

for doc_id, doc, distance in zip(
    results["ids"][0],
    results["documents"][0],
    results["distances"][0]
):
    print(f"ID: {doc_id}")
    print(f"Distance: {distance:.4f}")
    print(f"Content: {doc}")
    print("-" * 60)

Semantic Search Results:

ID: study02
Distance: 0.4433
Content: SQL JOIN operations combine records from multiple database tables.
------------------------------------------------------------
ID: study03
Distance: 1.8255
Content: Machine learning algorithms learn patterns from historical data.
------------------------------------------------------------
ID: study05
Distance: 1.9426
Content: Python lists store multiple values in an ordered and changeable collection.
------------------------------------------------------------


In [14]:
results = collection.query(
    query_texts=[
        "How do computers learn patterns from examples?"
    ],
    n_results=2
)

print("AI / ML Search Results:\n")

for doc_id, doc, distance in zip(
    results["ids"][0],
    results["documents"][0],
    results["distances"][0]
):
    print(f"ID: {doc_id}")
    print(f"Distance: {distance:.4f}")
    print(f"Content: {doc}")
    print("-" * 60)

AI / ML Search Results:

ID: study03
Distance: 0.7827
Content: Machine learning algorithms learn patterns from historical data.
------------------------------------------------------------
ID: study01
Distance: 1.2926
Content: JavaScript functions allow developers to organize reusable blocks of code.
------------------------------------------------------------


In [15]:
results = collection.query(
    query_texts=[
        "How do computers learn patterns from examples?"
    ],
    n_results=1
)

print(results["documents"][0])

['Machine learning algorithms learn patterns from historical data.']


## READ — Semantic Search with Metadata Filtering

Use `where` to restrict semantic search to documents
with a specific metadata value.

In [16]:
results = collection.query(
    query_texts=[
        "programming concepts for someone who is just starting"
    ],
    n_results=3,
    where={"level": "beginner"}
)

print("Beginner-Level Search Results:\n")

for doc_id, doc, metadata, distance in zip(
    results["ids"][0],
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print(f"ID: {doc_id}")
    print(f"Subject: {metadata['subject']}")
    print(f"Level: {metadata['level']}")
    print(f"Distance: {distance:.4f}")
    print(f"Content: {doc}")
    print("-" * 60)

Beginner-Level Search Results:

ID: study01
Subject: javascript
Level: beginner
Distance: 1.4810
Content: JavaScript functions allow developers to organize reusable blocks of code.
------------------------------------------------------------
ID: study04
Subject: react
Level: beginner
Distance: 1.6458
Content: React components help developers build reusable user interface elements.
------------------------------------------------------------
ID: study05
Subject: python
Level: beginner
Distance: 1.9369
Content: Python lists store multiple values in an ordered and changeable collection.
------------------------------------------------------------


In [17]:
results = collection.query(
    query_texts=[
        "How can I store multiple values together?"
    ],
    n_results=2,
    where={"subject": "python"}
)

print("Python Search Results:\n")

for doc_id, doc, metadata, distance in zip(
    results["ids"][0],
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print(f"ID: {doc_id}")
    print(f"Subject: {metadata['subject']}")
    print(f"Distance: {distance:.4f}")
    print(f"Content: {doc}")
    print("-" * 60)

Python Search Results:

ID: study05
Subject: python
Distance: 0.9717
Content: Python lists store multiple values in an ordered and changeable collection.
------------------------------------------------------------


In [18]:
collection.query(
    query_texts=["programming concepts"],
    n_results=3
)

{'ids': [['study01', 'study03', 'study06']],
 'embeddings': None,
 'documents': [['JavaScript functions allow developers to organize reusable blocks of code.',
   'Machine learning algorithms learn patterns from historical data.',
   'REST APIs allow applications to communicate using HTTP requests.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'subject': 'javascript', 'level': 'beginner'},
   {'level': 'intermediate', 'subject': 'machine_learning'},
   {'subject': 'backend', 'level': 'intermediate'}]],
 'distances': [[1.3073949813842773, 1.5655994415283203, 1.5747976303100586]]}

## UPDATE — Modify Existing Study Resource

Update an existing document using its unique ID.

We will modify the SQL resource and add an `updated` metadata field.

In [20]:
collection.update(
    ids=["study02"],

    documents=[
        "SQL JOIN operations combine records from multiple database tables and are useful for relational data analysis."
    ],

    metadatas=[
        {
            "subject": "sql",
            "level": "intermediate",
            "updated": True
        }
    ]
)

print("Study resource updated successfully!")

Study resource updated successfully!


In [21]:
updated = collection.get(ids=["study02"])

print("Updated Document:")
print("ID:", updated["ids"][0])
print("Content:", updated["documents"][0])
print("Metadata:", updated["metadatas"][0])

Updated Document:
ID: study02
Content: SQL JOIN operations combine records from multiple database tables and are useful for relational data analysis.
Metadata: {'subject': 'sql', 'updated': True, 'level': 'intermediate'}


In [22]:
collection.upsert(
    ids=["study03", "study07"],
    documents=[
        "Machine learning models identify patterns from data using statistical and computational techniques.",
        "Git helps developers track changes to source code and collaborate on software projects."
    ],
    metadatas=[
        {
            "subject": "machine_learning",
            "level": "intermediate",
            "updated": True
        },
        {
            "subject": "git",
            "level": "beginner"
        }
    ]
)

print("Upsert completed!")
print("Total documents:", collection.count())

Upsert completed!
Total documents: 7


In [23]:
result = collection.get(ids=["study03", "study07"])

print("Upsert Verification:\n")

for doc_id, doc, metadata in zip(
    result["ids"],
    result["documents"],
    result["metadatas"]
):
    print(f"ID: {doc_id}")
    print(f"Subject: {metadata['subject']}")
    print(f"Level: {metadata['level']}")
    print(f"Content: {doc}")
    print("-" * 60)

Upsert Verification:

ID: study03
Subject: machine_learning
Level: intermediate
Content: Machine learning models identify patterns from data using statistical and computational techniques.
------------------------------------------------------------
ID: study07
Subject: git
Level: beginner
Content: Git helps developers track changes to source code and collaborate on software projects.
------------------------------------------------------------


In [24]:
collection.delete(ids=["study01"])

print("study01 deleted successfully!")
print("Remaining documents:", collection.count())

study01 deleted successfully!
Remaining documents: 6


In [25]:
result = collection.get(ids=["study01"])

print("Search result for study01:")
print(result)

Search result for study01:
{'ids': [], 'embeddings': None, 'documents': [], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': []}


In [26]:
collection.delete(
    where={"level": "beginner"}
)

print("All beginner-level resources deleted!")
print("Remaining documents:", collection.count())

All beginner-level resources deleted!
Remaining documents: 3


In [27]:
result = collection.get()

print("Remaining StudyVault documents:\n")

for doc_id, doc, metadata in zip(
    result["ids"],
    result["documents"],
    result["metadatas"]
):
    print(f"ID: {doc_id}")
    print(f"Subject: {metadata['subject']}")
    print(f"Level: {metadata['level']}")
    print(f"Content: {doc}")
    print("-" * 60)

Remaining StudyVault documents:

ID: study02
Subject: sql
Level: intermediate
Content: SQL JOIN operations combine records from multiple database tables and are useful for relational data analysis.
------------------------------------------------------------
ID: study03
Subject: machine_learning
Level: intermediate
Content: Machine learning models identify patterns from data using statistical and computational techniques.
------------------------------------------------------------
ID: study06
Subject: backend
Level: intermediate
Content: REST APIs allow applications to communicate using HTTP requests.
------------------------------------------------------------


In [28]:
where={"level": "beginner"}

In [29]:
print("Final document count:", collection.count())

Final document count: 3


In [30]:
final_state = collection.get()

print("FINAL STUDYVAULT STATE\n")
print("=" * 70)

for doc_id, doc, metadata in zip(
    final_state["ids"],
    final_state["documents"],
    final_state["metadatas"]
):
    print(f"ID: {doc_id}")
    print(f"Subject: {metadata['subject']}")
    print(f"Level: {metadata['level']}")
    print(f"Content: {doc}")
    print("-" * 70)

FINAL STUDYVAULT STATE

ID: study02
Subject: sql
Level: intermediate
Content: SQL JOIN operations combine records from multiple database tables and are useful for relational data analysis.
----------------------------------------------------------------------
ID: study03
Subject: machine_learning
Level: intermediate
Content: Machine learning models identify patterns from data using statistical and computational techniques.
----------------------------------------------------------------------
ID: study06
Subject: backend
Level: intermediate
Content: REST APIs allow applications to communicate using HTTP requests.
----------------------------------------------------------------------


In [31]:
print("Remaining IDs:", final_state["ids"])

Remaining IDs: ['study02', 'study03', 'study06']


## Practice Exercises

Try the following exercises to practice ChromaDB CRUD operations using the StudyVault collection.

### Exercise 1 — Add New Study Resources

Add at least 5 new study resources to the `student_knowledge` collection.

Each resource should contain:

- A unique ID
- A document
- A subject
- A difficulty level

Example subjects:

- Java
- Python
- SQL
- React
- Machine Learning

---

### Exercise 2 — Practice UPSERT

Use `upsert()` to:

1. Update two existing study resources.
2. Add one completely new study resource.
3. Verify the changes using `get()`.

---

### Exercise 3 — Practice Metadata Filtering

Use a `where` filter to retrieve only resources belonging to a particular subject.

For example:

- Retrieve only Python resources.
- Retrieve only SQL resources.
- Retrieve only beginner resources.

---

### Exercise 4 — Practice Semantic Search

Use `query()` to search for study resources based on meaning.

Try questions such as:

- "How can I manipulate data?"
- "How do websites communicate with servers?"
- "How can I build reusable UI components?"
- "How do machines learn from data?"

Observe which documents ChromaDB returns.

---

### Exercise 5 — Practice DELETE

Choose a metadata condition and delete all matching resources.

Then use:

```python
collection.count()

In [32]:
collection.get()

{'ids': ['study02', 'study03', 'study06'],
 'embeddings': None,
 'documents': ['SQL JOIN operations combine records from multiple database tables and are useful for relational data analysis.',
  'Machine learning models identify patterns from data using statistical and computational techniques.',
  'REST APIs allow applications to communicate using HTTP requests.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'subject': 'sql', 'updated': True, 'level': 'intermediate'},
  {'updated': True, 'subject': 'machine_learning', 'level': 'intermediate'},
  {'level': 'intermediate', 'subject': 'backend'}]}